# RECIPE R-ES-003: Analisis Resistivitas Geolistrik untuk Interpretasi Material Bawah Permukaan
**Status**: Draft / Dry-run  
**Target Environment**: Python 3.8+ (`pandas`, `numpy`, `matplotlib`)  

---

## 1. Problem Definition
Data resistivitas geolistrik dapat digunakan untuk mempelajari variasi sifat kelistrikan material bawah permukaan. Analisis ini bertujuan mengidentifikasi zona dengan nilai resistivitas berbeda, melihat perubahan resistivitas terhadap kedalaman, dan memperkirakan kemungkinan jenis material penyusun bawah permukaan.

*Catatan Penting*: Interpretasi tidak boleh hanya berdasarkan nilai resistivitas. Kondisi geologi, kandungan air, porositas, salinitas, dan jenis batuan juga memengaruhi hasil pengukuran.

---

## 2. Minimal Theory & Equations
Resistivitas listrik merupakan ukuran kemampuan suatu material dalam menghambat aliran arus listrik, dinyatakan dalam ohm meter ($\Omega\cdot\text{m}$).

$$\rho = R \frac{A}{L}$$

- $\rho$ = resistivitas material ($\Omega\cdot\text{m}$)
- $R$ = hambatan listrik ($\Omega$)
- $A$ = luas penampang ($\text{m}^2$)
- $L$ = panjang material ($\text{m}$)

Dalam metode geolistrik, resistivitas semu diperoleh dari pengukuran beda potensial dan arus dengan memperhitungkan faktor geometri susunan elektroda. Nilai resistivitas semu bukan selalu resistivitas sebenarnya dari satu lapisan, karena pengukuran dapat dipengaruhi oleh beberapa lapisan sekaligus.

---

## 3. Input & Constraints
Data yang diperlukan berupa file CSV yang memuat:
- `x`: Koordinat horizontal X
- `y`: Koordinat horizontal Y
- `depth_m`: Kedalaman pengukuran (m)
- `resistivity_ohm_m`: Nilai resistivitas ($\Omega\cdot\text{m}$)

**Ketentuan analisis**:
1. Nilai resistivitas harus lebih besar dari zero ($>0$).
2. Data kosong dan nilai tidak valid harus diperiksa.
3. Satuan kedalaman dan resistivitas harus konsisten.
4. Interpretasi jenis material hanya berupa kemungkinan, bukan kepastian.
5. Jika data merupakan resistivitas semu, hasil tidak boleh langsung dianggap sebagai model resistivitas sebenarnya tanpa proses interpretasi atau inversi yang sesuai.

In [ ]:
# Cell 1: Setup Folder Kerja & Generator Data Sampel Geolistrik Dummy
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

file_path = "data_resistivitas.csv"
output_dir = "hasil_analisis"
os.makedirs(output_dir, exist_ok=True)

# Membuat file data dummy jika belum ada
if not os.path.exists(file_path):
    dummy_data = {
        "x": [10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0],
        "y": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        "depth_m": [1.5, 3.0, 5.0, 8.0, 12.0, 18.0, 25.0],
        "resistivity_ohm_m": [8.5, 25.4, 65.0, 180.0, 450.0, 1200.0, 850.0]
    }
    pd.DataFrame(dummy_data).to_csv(file_path, index=False)
    print(f"File sampel berhasil dibuat: {file_path}")
else:
    print(f"Menggunakan file data yang ada: {file_path}")

In [ ]:
# Cell 2: Membaca, Validasi, dan Pengelompokan Interpretasi Geolistrik
df = pd.read_csv(file_path)

# Memeriksa kolom yang diperlukan
required = ["x", "y", "depth_m", "resistivity_ohm_m"]
missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(f"Kolom belum tersedia: {missing}")

# Mengubah data menjadi numerik
for col in required:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Memeriksa data kosong
df = df.dropna(subset=required).copy()

# Menghapus nilai resistivitas tidak valid
df = df[df["resistivity_ohm_m"] > 0]
df = df[df["depth_m"] >= 0]

if df.empty:
    raise ValueError("Tidak ada data valid untuk dianalisis.")

# Klasifikasi awal untuk eksplorasi
def interpretasi(rho):
    if rho < 10:
        return "Sangat rendah: kemungkinan material konduktif"
    elif rho < 100:
        return "Rendah-menengah: kemungkinan material lembap/lempungan"
    elif rho < 1000:
        return "Menengah-tinggi: kemungkinan pasir, kerikil, atau batuan"
    else:
        return "Tinggi: kemungkinan material sangat resisif"

df["interpretasi_awal"] = df["resistivity_ohm_m"].apply(interpretasi)

# Menyimpan tabel hasil
hasil_csv = os.path.join(output_dir, "hasil_interpretasi.csv")
df.to_csv(hasil_csv, index=False)

# Statistik ringkas
print("=== STATISTIK RINGKAS DATA GEOLISTRIK ===")
print("Jumlah data valid:", len(df))
print("Resistivitas minimum:", df["resistivity_ohm_m"].min(), "ohm meter")
print("Resistivitas maksimum:", df["resistivity_ohm_m"].max(), "ohm meter")
print("Resistivitas rata-rata:", round(df["resistivity_ohm_m"].mean(), 2), "ohm meter")

print("\n=== TABEL HASIL INTERPRETASI AWAL ===")
print(df[["depth_m", "resistivity_ohm_m", "interpretasi_awal"]].to_string(index=False))

In [ ]:
# Cell 3: Visualisasi Grafik Resistivitas terhadap Kedalaman
plt.figure(figsize=(8, 6))

plt.scatter(df["resistivity_ohm_m"], df["depth_m"], color="blue", s=50, zorder=3)
plt.plot(df["resistivity_ohm_m"], df["depth_m"], color="darkblue", linestyle="--", alpha=0.7)

plt.xscale("log")
plt.gca().invert_yaxis()  # Kedalaman bertambah ke arah bawah
plt.xlabel("Resistivitas (ohm meter) - Skala Logaritma")
plt.ylabel("Kedalaman (m)")
plt.title("Profil Hubungan Resistivitas dengan Kedalaman")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()

grafik_path = os.path.join(output_dir, "grafik_resistivitas.png")
plt.savefig(grafik_path, dpi=300)
plt.show()

print(f"\nFile hasil CSV dan Grafik berhasil disimpan di folder: '{output_dir}'")

## 4. Optional Check and Verification
Sebelum hasil digunakan dalam laporan, lakukan pemeriksaan berikut:
- Pastikan seluruh kolom dan satuan data benar (`depth_m` dalam meter, `resistivity_ohm_m` dalam $\Omega\cdot\text{m}$).
- Periksa data kosong, nilai nol, dan nilai ekstrem.
- Pastikan sumbu kedalaman pada grafik meningkat ke arah bawah (`invert_yaxis`).
- Bandingkan interpretasi dengan informasi litologi atau kondisi geologi setempat jika tersedia.
- Bedakan resistivitas semu dari resistivitas hasil inversi.
- Hindari menyatakan keberadaan air tanah atau jenis batuan tertentu sebagai kesimpulan pasti tanpa bukti pendukung.